# Giai đoạn khám phá - tiền xử lý dữ liệu - trích xuất đặc trưng

Notebook này thực hiện quy trình trích xuất đặc trưng chuyên sâu để phân biệt văn bản do người viết và AI viết trong tiếng Việt.

- **Đầu vào:** File CSV chứa cột văn bản (`text`) và nhãn phân loại (`label`).
- **Đầu ra:** Bộ dữ liệu tích hợp các đặc trưng số học (features) phục vụ huấn luyện mô hình.

### Pipeline gồm 7 giai đoạn chính:
- **Khám phá:** Thống kê cơ bản và kiểm tra chất lượng dữ liệu thô.
- **Chuẩn hóa & Lọc:** Xử lý Unicode, nhận diện ngôn ngữ và lọc nhiễu.
- **Lấy mẫu:** Ép cân bằng nhãn 50/50 và phân tầng theo độ dài văn bản.

#### Phân tách pipeline trích xuất đặc trưng ML
- **Logic:** Trích xuất mật độ từ nối và đo lường tính logic của lập luận.
- **Xác suất (GPT-Neo):** Tính toán độ bất ngờ (Surprisal) và biến thiên xác suất từ.
- **Cấu trúc:** Phân tích nhịp điệu câu, dấu câu và thói quen dùng đại từ.
- **Ngữ nghĩa (PhoBERT):** Đo lường độ mạch lạc ngữ nghĩa và dòng chảy chủ đề.

#### Phân tách pipeline cho DL
- Thực hiện word_tokenize text bằng underthesea

#### Tổng kết
- Lưu file và đẩy lên Hugging Face
- Lưu check-point mỗi lần xong một bước pipeline

In [ ]:
!pip install python-dotenv huggingface_hub datasets underthesea langdetect pandarallel -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.5 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

In [ ]:
import gc
import glob
import hashlib
import json
import os
import re
import string
import unicodedata
from collections import Counter
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login
from langdetect import LangDetectException, detect
from pandarallel import pandarallel
from scipy.stats import entropy, kurtosis, skew
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoTokenizer
)
from underthesea import (
    ner,
    pos_tag,
    sent_tokenize,
    word_tokenize
)
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 1 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [ ]:
env_path = find_dotenv()

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

In [ ]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

### Giai đoạn 1: Khám phá cơ bản và tiền xử lý

Phần này thiết lập các bước nền tảng để làm sạch và chuẩn bị dữ liệu chất lượng cao:

1.  **PipelineStep**: Lớp trừu tượng định nghĩa phương thức `execute`, đảm bảo tính nhất quán cho toàn bộ hệ thống.
2.  **BasicExplorationStep**: Thực hiện thống kê mô tả, kiểm tra dữ liệu thiếu, trùng lặp và phân bổ nhãn ban đầu.
3.  **SafeNormalizationAndFilteringStep**:
    - Chuẩn hóa Unicode (NFC) và loại bỏ khoảng trắng thừa.
    - Lọc trùng lặp bằng mã băm MD5.
    - Loại bỏ văn bản kém chất lượng (spam, ký tự lặp, tỷ lệ chữ cái thấp).
    - Kiểm tra ngôn ngữ tiếng Việt bằng `langdetect`.
    - Giới hạn độ dài văn bản (mặc định 150-600 từ) dựa trên `underthesea`.
4.  **StratifiedSamplingStep**:
    - Thực hiện Undersampling để ép tỷ lệ nhãn về 50/50.
    - **Phân tầng (Stratify) theo độ dài**: Đảm bảo phân phối chiều dài văn bản giữa Người và AI là như nhau, tránh việc mô hình bị đánh lừa bởi độ dài câu.

In [ ]:
class PipelineStep:
    def execute(self, df, text_col, label_col):
        raise NotImplementedError("Phải triển khai phương thức execute")

In [ ]:
class BasicExplorationStep(PipelineStep):
    def execute(self, df, text_col, label_col):
        print("\n" + "="*60)
        print(" GIAI ĐOẠN 1: KHÁM PHÁ DỮ LIỆU CƠ BẢN")
        print("="*60)

        total = len(df)

        print("\n--- 1. TỔNG QUAN DATASET ---")
        missing_text = df[text_col].isnull().sum()
        empty_text = (df[text_col].dropna().astype(str).str.strip() == '').sum()
        dupes = df.duplicated(subset=[text_col]).sum()

        print(f"Tổng số mẫu             : {total:,} dòng")
        print(f"Mẫu thiếu text (NaN)   : {missing_text:,} dòng ({missing_text/total:.2%})")
        print(f"Mẫu rỗng/khoảng trắng  : {empty_text:,} dòng ({empty_text/total:.2%})")
        print(f"Mẫu trùng lặp text     : {dupes:,} dòng ({dupes/total:.2%})")

        print("\n--- 2. PHÂN PHỐI CHIỀU DÀI VĂN BẢN ---")
        valid_texts = df[text_col].dropna().astype(str)
        valid_texts = valid_texts[valid_texts.str.strip() != '']

        if not valid_texts.empty:
            char_lengths = valid_texts.apply(len)
            word_lengths = valid_texts.apply(lambda x: len(x.split()))

            print(f"[Số lượng KÝ TỰ (Characters)]")
            print(f"  - Trung bình : {char_lengths.mean():.0f}")
            print(f"  - Trung vị   : {char_lengths.median():.0f}")
            print(f"  - Min / Max  : {char_lengths.min()} / {char_lengths.max()}")

            print(f"\n[Số lượng TỪ (Words)]")
            print(f"  - Trung bình : {word_lengths.mean():.0f}")
            print(f"  - Trung vị   : {word_lengths.median():.0f}")
            print(f"  - Min / Max  : {word_lengths.min()} / {word_lengths.max()}")
        else:
            print("Không có dữ liệu văn bản hợp lệ để thống kê.")

        if label_col in df.columns:
            print("\n--- 3. PHÂN BỔ NHÃN ---")
            missing_labels = df[label_col].isnull().sum()
            if missing_labels > 0:
                print(f"Có {missing_labels:,} dòng bị thiếu nhãn!\n")

            label_counts = df[label_col].value_counts()
            label_props = df[label_col].value_counts(normalize=True) * 100

            dist_df = pd.DataFrame({
                'Số lượng': label_counts,
                'Tỷ lệ (%)': label_props.round(2)
            })
            print(dist_df.to_string())

            if len(label_counts) > 1:
                majority = label_counts.max()
                minority = label_counts.min()
                imbalance_ratio = majority / minority
                print(f"\nTỷ lệ chênh lệch nhãn: {imbalance_ratio:.2f} lần")
                if imbalance_ratio > 3:
                    print("   (Dữ liệu có dấu hiệu mất cân bằng, cần xử lý")

        print("="*60)

        return df

In [ ]:
class SafeNormalizationAndFilteringStep(PipelineStep):
    def __init__(self, min_words=150, max_words=600):
        self.min_words = min_words
        self.max_words = max_words

    def _normalize_text(self, text):
        if not isinstance(text, str): return ""
        text = unicodedata.normalize('NFC', text)
        text = re.sub(r'[\r\n]+', ' ', text)
        text = re.sub(r'\s{2,}', ' ', text)
        return text.strip()

    def _get_hash(self, text):
        return hashlib.md5(text.encode('utf-8')).hexdigest()

    def _count_words(self, text):
        try:
            return len(word_tokenize(text))
        except Exception:
            return 0

    def _is_vietnamese(self, text):
        try:
            return detect(text) == 'vi'
        except LangDetectException:
            return False

    def _is_high_quality(self, text):
        if not text: return False

        if re.search(r'<.*?>', text):
            return False

        if re.search(r'(.)\1{4,}', text):
            return False

        total_chars = len(text)
        alpha_chars = sum(c.isalpha() for c in text)
        if total_chars > 0 and (alpha_chars / total_chars) < 0.65:
            return False

        return True

    def execute(self, df, text_col, label_col=None):
        print("\n" + "="*60)
        print("GIAI ĐOẠN 2: CHUẨN HÓA VÀ LỌC NHIỄU")
        print("="*60)
        initial_count = len(df)

        df = df.dropna(subset=[text_col]).copy()

        print("[1/5] Chuẩn hóa Unicode và tạo Hash MD5...")
        df[text_col] = df[text_col].apply(self._normalize_text)
        df = df[df[text_col] != ""]
        df['text_hash'] = df[text_col].apply(self._get_hash)

        print("[2/5] Lọc trùng lặp (Hashing Deduplication)...")
        dupes_count = df.duplicated(subset=['text_hash']).sum()
        df = df.drop_duplicates(subset=['text_hash'], keep='first')
        df = df.drop(columns=['text_hash'])
        print(f"Đã loại bỏ {dupes_count:,} mẫu trùng lặp.")

        print("[3/5] Lọc chất lượng (Spam, ký tự lặp, tỷ lệ alphabet)...")
        quality_mask = df[text_col].apply(self._is_high_quality)
        count_before_quality = len(df)
        df = df[quality_mask]
        print(f"Đã loại bỏ {count_before_quality - len(df):,} mẫu kém chất lượng.")

        print("[4/5] Kiểm tra ngôn ngữ (langdetect)...")
        count_before_lang = len(df)
        vi_mask = df[text_col].apply(self._is_vietnamese)
        df = df[vi_mask]
        print(f"Đã loại bỏ {count_before_lang - len(df):,} mẫu không phải Tiếng Việt/không hợp lệ.")

        print(f"[5/5] Phân tách từ (underthesea) & Lọc độ dài ({self.min_words}-{self.max_words})...")
        word_counts = df[text_col].apply(self._count_words)
        count_before_len = len(df)
        df = df[(word_counts >= self.min_words) & (word_counts <= self.max_words)]
        print(f"Đã loại bỏ {count_before_len - len(df):,} mẫu vi phạm độ dài.")

        print("-" * 60)
        print(f"Hoàn tất. Mẫu hợp lệ: {len(df):,} (Từ gốc {initial_count:,})")
        return df.reset_index(drop=True)

### Bước tiền xử lý cuối là StratifiedSamplingStep

Bước này giúp mô hình **không bị "đánh lừa" bởi độ dài văn bản**:

*   **Cân bằng 50/50:** Đảm bảo dữ liệu không bị lệch về phía Người hay AI, giúp mô hình công tâm hơn.
*   **Chặn lỗi "học vẹt":** Nếu vô tình các bài AI luôn dài và bài Người luôn ngắn, mô hình sẽ lầm tưởng cứ dài là AI. Phân tầng (Stratified) giúp các nhóm độ dài (ngắn, trung bình, dài) ở cả hai bên Người và AI đều tương đương nhau, buộc mô hình phải soi kỹ vào nội dung và cấu trúc thay vì chỉ đếm chữ.

#### Cơ chế hoạt động:
1.  **Tokenization:** Sử dụng `word_tokenize` của Underthesea để xác định độ dài thực tế dựa trên đơn vị từ tiếng Việt (thay vì đếm khoảng trắng).
2.  **Binning (Phân nhóm):** Chia dữ liệu thành các nhóm độ dài dựa trên các phân vị (Quantiles). Điều này biến độ dài liên tục thành các danh mục rời rạc.
3.  **Stratified Undersampling:** Khi rút trích dữ liệu từ nhóm đa số, hệ thống thực hiện lấy mẫu có phân tầng (Stratify) theo các nhóm độ dài đã chia. Kết quả là tập dữ liệu cuối cùng có phân phối hình học của độ dài văn bản hoàn toàn trùng khớp giữa Người và AI.

In [ ]:
class StratifiedSamplingStep(PipelineStep):
    def __init__(self, seed=2026, n_bins=4):
        self.seed = seed
        self.n_bins = n_bins
        self.bin_edges_ = None

    def execute(self, df, text_col, label_col):
        print("\n" + "="*60)
        print("GIAI ĐOẠN 3: LẤY MẪU CÂN BẰNG & PHÂN TẦNG")
        print("="*60)

        if label_col not in df.columns:
            return df

        print("[1/3] Tính toán độ dài văn bản...")
        df['seg_length'] = df[text_col].parallel_apply(
            lambda x: len(word_tokenize(x, format="text").split())
        )

        print("[2/3] Phân nhóm (Binning) độ dài...")
        if self.bin_edges_ is None:
            _, bins = pd.qcut(df['seg_length'], q=self.n_bins, duplicates='drop', retbins=True)
            bins[0] = -np.inf
            bins[-1] = np.inf
            self.bin_edges_ = bins
            print(f" -> Đã thiết lập ranh giới Bins: {np.round(self.bin_edges_, 2)}")

        df['length_bin'] = pd.cut(df['seg_length'], bins=self.bin_edges_, labels=False)

        print("[3/3] Thực hiện Undersampling ép tỷ lệ 50/50...")
        class_counts = df[label_col].value_counts()
        minority_class = class_counts.idxmin()
        majority_class = class_counts.idxmax()
        target_size = class_counts[minority_class]

        df_minority = df[df[label_col] == minority_class].copy()
        df_majority = df[df[label_col] == majority_class].copy()

        bin_counts = df_majority['length_bin'].value_counts()
        has_rare_strata = (bin_counts < 2).any()

        n_bins_actual = len(bin_counts)
        discard_size = len(df_majority) - target_size

        is_safe_to_stratify = (
            not has_rare_strata and
            n_bins_actual > 1 and
            target_size >= n_bins_actual and
            discard_size >= n_bins_actual
        )

        if is_safe_to_stratify:
            df_sampled, _ = train_test_split(
                df_majority,
                train_size=target_size,
                stratify=df_majority['length_bin'],
                random_state=self.seed
            )
            print("Đã lấy mẫu phân tầng (Stratified) thành công trên class Đa số.")
        else:
            if discard_size < n_bins_actual:
                print(f"Khoảng cách label quá nhỏ nên phải bỏ qua phân tầng.")
            else:
                print("Phát hiện nhóm độ dài có dưới 2 mẫu trong label đa số.")
            print("Đã chuyển sang Random Undersampling để bảo toàn cấu trúc dữ liệu.")
            df_sampled = df_majority.sample(n=target_size, random_state=self.seed)

        df_final = pd.concat([df_minority, df_sampled])
        df_final = df_final.sample(frac=1, random_state=self.seed).drop(columns=['length_bin', 'seg_length'])

        print("-" * 60)
        print(f"Hoàn tất. Tỷ lệ: 50/50.")
        print(f"Tổng số mẫu: {len(df_final):,} (Mỗi class: {target_size:,})")
        return df_final.reset_index(drop=True)

## GIAI ĐOẠN: TRÍCH XUẤT ĐẶC TRƯNG (FEATURE EXTRACTION)

Sau khi đã có bộ dữ liệu sạch và cân bằng, chúng ta tiến hành trích xuất các đặc trưng định lượng để "số hóa" sự khác biệt giữa văn bản người và AI.

### Các nhóm đặc trưng trong giai đoạn này

1.  **Đặc trưng Logic & Lập luận (Logic):** AI thường lạm dụng từ nối để tạo cảm giác logic nhưng lại thiếu sự linh hoạt trong cách dẫn dắt ý tưởng.
2.  **Đặc trưng Xác suất (GPT-Neo):** AI được huấn luyện để chọn những từ có xác suất cao nhất (an toàn). Việc đo lường **Surprisal** (độ bất ngờ) giúp nhận diện những đoạn văn "quá mượt mà" một cách máy móc.
3.  **Đặc trưng Cấu trúc (Structural):** Con người có nhịp điệu viết thay đổi (câu dài, câu ngắn đan xen). AI thường tạo ra cấu trúc câu đều đặn hơn (biến thiên thấp).
4.  **Đặc trưng Ngữ nghĩa & Mạch lạc (PhoBERT):** Kiểm tra dòng chảy thực thể và sự nhất quán chủ đề từ đầu đến cuối văn bản, nơi mà AI đôi khi bị "trôi dạt" hoặc lặp ý vô nghĩa.

Việc kết hợp đa dạng các nhóm đặc trưng này giúp mô hình bắt bài được AI ở nhiều góc độ khác nhau thay vì chỉ dựa vào từ vựng đơn thuần.

In [ ]:
def load_const_seeds(filepath="Data/const_seeds.json"):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Lỗi không tìm thấy file cấu hình tại '{filepath}'.")

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"File '{filepath}' không chuẩn định dạng JSON. Chi tiết lỗi: {e}")
    except Exception as e:
        raise RuntimeError(f"Lỗi không xác định khi đọc file '{filepath}': {e}")

CONST_SEED = load_const_seeds()

In [ ]:
class AdvancedLogicFeatureExtractor(PipelineStep):
    def __init__(self,
                 seeds=None,
                 pronoun_list=None,
                 verb_noise_list=None,
                 ngram_range=(2, 6),
                 min_df=5,
                 max_df=0.7,
                 production_markers_path=None):

        self.groups = ['causal', 'contrast', 'additive', 'temporal']

        if not seeds:
          raise ValueError("Thiếu file seed")

        self.seeds = seeds

        self.pronoun_list = pronoun_list or {
            'tôi', 'ta', 'chúng ta', 'chúng tôi', 'mình', 'bạn', 'các bạn', 'họ', 'bọn họ', 'anh', 'chị', 'em', 'ông', 'bà', 'chú', 'bác', 'nó', 'hắn', 'tao', 'mày', 'tớ', 'tụi', 'tụi nó', 'bọn', 'bọn nó', 'anh ấy', 'chị ấy', 'cô ấy', 'ông ấy', 'bà ấy', 'anh ta', 'chị ta', 'họ ta', 'cô', 'dì', 'dượng', 'cậu', 'mợ', 'thầy', 'con', 'cháu', 'người ta', 'ai đó', 'mọi người', 'chúng nó', 'tụi mình', 'bọn mình'
        }

        self.verb_noise_list = verb_noise_list or {
            'nghĩ', 'thấy', 'cho rằng', 'rằng', 'là', 'sẽ', 'đã', 'đang'
        }

        self.ngram_range = ngram_range
        self.min_df = min_df
        self.max_df = max_df

        self.dynamic_markers = {g: set() for g in self.groups}
        self.dynamic_blacklist = set()
        self.regex_patterns = {}

        self.production_markers_path = production_markers_path
        if self.production_markers_path:
            self._load_production_markers()
            self._compile_regex_from_seeds()

    def _is_noise(self, ngram):
        for p in self.pronoun_list:
            for v in self.verb_noise_list:
                if f"{p} {v}" in ngram:
                    return True

        if len(ngram.split()) > 6:
            return True

        return False

    def _build_dynamic_markers_in_memory(self, df, text_col):
        print("1. Đang cắt đầu câu (Sentence Starters)...")
        starters = []
        for text in df[text_col].dropna():
            sentences = sent_tokenize(str(text))
            for sent in sentences:
                words = sent.split()
                if len(words) >= 2:
                    starters.append(sent.lower())

        if not starters:
            print("Dữ liệu trống. Sẽ sử dụng Seeds mặc định.")
            self._compile_regex_from_seeds()
            return

        print("2. Đang tạo N-gram và đếm tần suất...")
        vectorizer = CountVectorizer(ngram_range=self.ngram_range, min_df=self.min_df, max_df=self.max_df)
        X = vectorizer.fit_transform(starters)
        freqs = zip(vectorizer.get_feature_names_out(), X.sum(axis=0).tolist()[0])
        sorted_ngrams = sorted(freqs, key=lambda x: x[1], reverse=True)

        print("3. Khởi chạy Dynamic Noise Filter và phân loại (RAM)...")
        for ngram, freq in sorted_ngrams:
            if self._is_noise(ngram):
                self.dynamic_blacklist.add(ngram)
                continue

            for g, seed_words in self.seeds.items():
                if any(s in ngram for s in seed_words):
                    self.dynamic_markers[g].add(ngram)
                    break

        self._compile_regex_from_seeds()

        print(f"Hoàn tất lưu pipeline N-gram vào bộ nhớ.")
        print(f"Đã lọc tự động {len(self.dynamic_blacklist)} cụm nhiễu.")

    def _compile_regex_from_seeds(self):
        for g in self.groups:
            manual_seeds = set(self.seeds.get(g, []))
            dynamic_found = self.dynamic_markers[g]

            all_markers = manual_seeds.union(dynamic_found)

            if not all_markers:
                continue

            sorted_markers = sorted(list(all_markers), key=len, reverse=True)
            pattern = r'(?<!\w)(' + '|'.join(map(re.escape, sorted_markers)) + r')(?!\w)'
            self.regex_patterns[g] = re.compile(pattern, re.IGNORECASE | re.UNICODE)

    def _extract_logic_metrics(self, text):
        sentences = sent_tokenize(str(text))
        num_sentences = len(sentences)

        res = {f'{g}_count': 0 for g in self.groups}
        res['transition_count'] = 0

        if num_sentences == 0:
            return pd.Series(res)

        for i, sent in enumerate(sentences):
            sent_lower = sent.lower()
            is_transition = False

            for g in self.groups:
                if g not in self.regex_patterns:
                    continue

                pattern = self.regex_patterns[g]
                matches = list(pattern.finditer(sent_lower))
                count = len(matches)

                if count > 0:
                    res[f'{g}_count'] += count

                    if not is_transition and matches[0].start() <= 20:
                        is_transition = True

            if is_transition:
                res['transition_count'] += 1

        res['num_sentences'] = num_sentences
        return pd.Series(res)

    def export_markers_for_production(self, expected_logic_val, filepath="logic_markers_prod.json"):
        export_data = {
            "markers": {k: list(v) for k, v in self.dynamic_markers.items()},
            "expected_logic": float(expected_logic_val)
        }
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(export_data, f, ensure_ascii=False, indent=4)
        print(f"\nĐã xuất cấu hình sang môi trường Production: {filepath}")

    def _load_production_markers(self):
        try:
            with open(self.production_markers_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if "markers" in data:
                self.dynamic_markers = {k: set(v) for k, v in data["markers"].items()}
                self.saved_expected_logic = data.get("expected_logic", 0.0)
            else:
                self.dynamic_markers = {k: set(v) for k, v in data.items()}
                self.saved_expected_logic = 0.0

            print(f"Đã nạp thành công N-gram & Mean từ EDA: {self.production_markers_path}")
        except Exception as e:
            print(f"Lỗi nạp file N-gram Production: {e}")
            self.saved_expected_logic = 0.0

    def execute(self, df, text_col, label_col=None):
        print("\n" + "="*60)
        print("GIAI ĐOẠN 4: KHAI PHÁ N-GRAM & TRÍCH XUẤT ĐẶC TRƯNG LOGIC")
        print("="*60)

        if not self.production_markers_path:
            self._build_dynamic_markers_in_memory(df, text_col)

        if 'seg_length' not in df.columns:
            df['seg_length'] = df[text_col].apply(lambda x: len(str(x).split()))

        try:
            tqdm.pandas(desc="[Logic] Áp dụng Regex tính Metrics")
            metrics_df = df[text_col].progress_apply(self._extract_logic_metrics)
        except (AttributeError, NameError):
            metrics_df = df[text_col].apply(self._extract_logic_metrics)

        df = pd.concat([df, metrics_df], axis=1)
        for g in self.groups:
            df[f'{g}_density'] = df[f'{g}_count'] / (df['seg_length'] + 1e-6)

        df['logic_transition_points'] = df['transition_count'] / (df['num_sentences'] + 1e-6)
        densities = df[[f'{g}_density' for g in self.groups]]
        df['logic_mismatch_score'] = densities.std(axis=1)
        df['logic_density'] = sum(df[f'{g}_count'] for g in self.groups) / (df['seg_length'] + 1e-6)
        if self.production_markers_path:
            expected_logic = getattr(self, 'saved_expected_logic', 0.0)
        else:
            expected_logic = float(df['logic_density'].mean())
            self.export_markers_for_production(expected_logic, "Data/logic_markers_prod.json")

        df['logic_deviation'] = abs(df['logic_density'] - expected_logic)

        df = df.drop(columns=['logic_transition_points','num_sentences', 'transition_count'] + [f'{g}_count' for g in self.groups])

        return df

### 2. Đặc trưng Xác suất & Độ bất ngờ (CausalSurprisalStep)

**Mục đích:** Con người thường dùng từ ngữ linh hoạt và đôi khi khó đoán. Ngược lại, AI có xu hướng chọn những từ có xác suất cao nhất, tạo ra văn bản "quá mượt mà".

**Cơ chế:** Tính toán **Surprisal** (độ bất ngờ). Văn bản có độ biến thiên xác suất thấp thường là dấu hiệu của máy viết.

In [ ]:
class CausalSurprisalStep(PipelineStep):
    def __init__(self, model_name="VietAI/gpt-neo-1.3B-vietnamese-news"):
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    def safe(self, v):
        return 0.0 if np.isnan(v) or np.isinf(v) else float(v)

    def _aggregate_to_word_level(self, surprisal_array, word_ids):
        if len(surprisal_array) != len(word_ids):
            return surprisal_array
        word_surprisals = {}
        for surp, w_id in zip(surprisal_array, word_ids):
            if w_id is not None:
                word_surprisals[w_id] = word_surprisals.get(w_id, 0) + surp
        if not word_surprisals or len(surprisal_array) == 0 or len(word_ids) == 0:
            return np.array([])
        return np.array([word_surprisals[k] for k in sorted(word_surprisals.keys())])

    def execute(self, df, text_col, label_col=None):
        print("\n" + "="*60)
        print(" GIAI ĐOẠN 5: CAUSAL SURPRISAL DYNAMICS (GPT-NEO)")
        print("="*60)

        feature_names = ['mean', 'var', 'tail', 'autocorr', 'skew', 'kurtosis', 'd2_var', 'd2_entropy']
        metrics = {k: np.zeros(len(df)) for k in feature_names}

        print(f"[Causal LM] Đang tải lên VRAM: {self.model_name}...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(self.model_name, use_fast=True)
            model = AutoModelForCausalLM.from_pretrained(self.model_name, torch_dtype=self.dtype)
            model.to(self.device)
            model.eval()
        except Exception as e:
            print(f"Lỗi tải model {self.model_name}: {e}")
            return df

        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')

        with torch.no_grad():
            for idx, text in enumerate(tqdm(df[text_col], desc="[GPT-Neo] Quét Causal")):
                try:
                    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(self.device)
                    input_ids = inputs["input_ids"]

                    if input_ids.size(1) < 4:
                        continue

                    outputs = model(input_ids)
                    shift_logits = outputs.logits[..., :-1, :].contiguous()
                    shift_labels = input_ids[..., 1:].contiguous()

                    surprisal_tensor = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
                    S_raw = surprisal_tensor.cpu().numpy().astype(np.float64)

                    word_ids = inputs.word_ids(batch_index=0)[1:]
                    S = self._aggregate_to_word_level(S_raw, word_ids)

                    if len(S) < 5 or np.std(S) < 1e-6:
                        continue

                    metrics['mean'][idx] = np.mean(S)
                    metrics['var'][idx] = np.var(S)
                    metrics['tail'][idx] = np.percentile(S, 95)
                    metrics['skew'][idx] = self.safe(skew(S))
                    metrics['kurtosis'][idx] = self.safe(kurtosis(S))

                    if len(S) > 2 and np.std(S) > 1e-6:
                        val = np.corrcoef(S[:-1], S[1:])[0,1]
                        metrics['autocorr'][idx] = 0.0 if np.isnan(val) else val

                    d1 = np.diff(S)
                    d2 = np.diff(d1)

                    if len(d2) > 10:
                        hist, _ = np.histogram(d2, bins=10, density=True)
                        metrics['d2_entropy'][idx] = entropy(hist + 1e-9)

                    metrics['d2_var'][idx] = np.var(d2) if len(d2) > 1 else 0.0

                except Exception as e:
                    pass

        print(f"[Memory] Giải phóng VRAM của {self.model_name}...")
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        for k in feature_names:
            df[f'gpt_{k}'] = metrics[k]

        return df

### 3. Đặc trưng cấu trúc & nhịp điệu (SyntacticAndStructuralStep)

**Mục đích:** Nhận diện "vân tay" trong cách sắp xếp câu và dấu câu. Người viết thường thay đổi độ dài câu tùy cảm xúc, trong khi AI tạo ra cấu trúc câu khá đồng đều.

**Cơ chế:** Đo lường sự biến thiên độ dài câu (CV), mật độ dấu phẩy và thói quen sử dụng đại từ nhân xưng.

In [ ]:
class SyntacticAndStructuralStep(PipelineStep):
    def __init__(self):
        raw_pronouns = [
            "tôi", "tao", "tớ", "mình", "ta",
            "chúng tôi", "chúng ta", "tụi tôi", "bọn tôi", "tụi mình",
            "bạn", "cậu", "mày", "mi",
            "các bạn", "tụi bây", "bọn mày",
            "nó", "hắn", "y", "người ta",
            "họ", "chúng nó", "tụi nó", "bọn họ",
            "ai", "gì", "nào", "bao nhiêu", "bao giờ"
        ]
        self.pronoun_set = set([p.replace(" ", "_") for p in raw_pronouns])

    def _compute_recombined_features(self, text):
        paragraphs = [p for p in text.split('\n') if p.strip()]

        sentences = []
        for p in paragraphs:
            sentences.extend(sent_tokenize(p))

        sent_words = [word_tokenize(s) for s in sentences]
        sent_lengths = [len(words) for words in sent_words]
        total_words = sum(sent_lengths)
        total_words_safe = max(1, total_words)

        if len(sent_lengths) > 1:
            d_k = np.abs(np.diff(sent_lengths))
            sentence_transition_var = float(np.var(d_k))
        else:
            sentence_transition_var = 0.0

        sent_len_mean = float(np.mean(sent_lengths)) if sent_lengths else 0.0
        sent_len_var = float(np.var(sent_lengths)) if sent_lengths else 0.0
        sent_len_cv = sent_len_var / (sent_len_mean + 1e-6)

        comma_counts = [s.count(',') for s in sentences]
        punct_counts = [sum(1 for char in s if char in string.punctuation) for s in sentences]

        comma_density = sum(comma_counts) / total_words_safe
        punctuation_density = sum(punct_counts) / total_words_safe

        comma_std_per_sentence = float(np.std(comma_counts)) if sentences else 0.0
        punctuation_std_per_sentence = float(np.std(punct_counts)) if sentences else 0.0

        all_words_lower = [w.lower() for words in sent_words for w in words]
        total_pronouns = sum(1 for w in all_words_lower if w in self.pronoun_set)
        pronoun_ratio = total_pronouns / total_words_safe

        return (
            sent_len_cv,
            sentence_transition_var,
            comma_density,
            punctuation_density,
            comma_std_per_sentence,
            punctuation_std_per_sentence,
            pronoun_ratio
        )

    def execute(self, df, text_col, label_col=None):
        print("\n" + "="*60)
        print(" GIAI ĐOẠN 6: STRUCTURAL IRREGULARITY & SYNTACTIC")
        print("="*60)
        tqdm.pandas(desc="[Struct/Syntactic] Trích xuất")
        res = df[text_col].progress_apply(self._compute_recombined_features)

        cols = [
            'sent_len_cv',
            'sentence_transition_var',
            'comma_density',
            'punctuation_density',
            'comma_std_per_sentence',
            'punctuation_std_per_sentence',
            'pronoun_ratio'
        ]
        df[cols] = pd.DataFrame(res.tolist(), index=df.index)
        return df

### 4. Đặc trưng Ngữ nghĩa & Mạch lạc (PhoBERTFeaturePipeline)

**Mục đích:** Kiểm tra sự nhất quán về nội dung. AI đôi khi bị "trôi dạt" chủ đề (Topic Drift) hoặc thiếu sự liên kết thực thể giữa các đoạn văn dài.

**Cơ chế:** Sử dụng **PhoBERT** để tính toán độ tương đồng cosine giữa các câu và đo lường độ mạch lạc ngữ nghĩa xuyên suốt văn bản.

In [ ]:
class PhoBERTFeaturePipeline(PipelineStep):
    def __init__(self, model_name="vinai/phobert-base-v2", batch_size=16):
        self.model_name = model_name
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = 256
        self.stride = 128

    def _compute_cpu_features(self, text):
        if not isinstance(text, str) or not text.strip():
            return (0.0, 0.0, [])

        try:
            sentences = sent_tokenize(text)

            try:
                tags = pos_tag(text)
                pos_counts = Counter(tag for _, tag in tags)
                total_tags = sum(pos_counts.values())
                if total_tags > 0:
                    probs = [c / total_tags for c in pos_counts.values()]
                    pos_var = float(np.var(probs))
                else: pos_var = 0.0
            except: pos_var = 0.0

            try:
                sentence_entities = []
                for s in sentences:
                    ner_words = [w for w, _, _, tg in ner(s) if tg != 'O']
                    noun_words = [w for w, tg in pos_tag(s) if str(tg).startswith('N')]
                    sentence_entities.append(set(ner_words + noun_words))

                overlaps = [len(set1 & set2) / max(1, len(set1 | set2))
                            for set1, set2 in zip(sentence_entities[:-1], sentence_entities[1:])]
                entity_grid = float(np.mean(overlaps)) if overlaps else 0.0
            except: entity_grid = 0.0

            return (pos_var, entity_grid, sentences)

        except Exception:
            return (0.0, 0.0, [])

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def _get_sentence_embeddings(self, sentences, tokenizer, model):
        all_sentence_embeddings = []

        chunked_inputs = []
        sentence_to_chunk_map = []

        current_chunk_idx = 0
        for sent in sentences:
            tokens = tokenizer(sent, add_special_tokens=False, return_attention_mask=False)['input_ids']

            if len(tokens) <= self.max_length - 2:
                chunk_tokens = [tokenizer.cls_token_id] + tokens + [tokenizer.sep_token_id]
                chunked_inputs.append(chunk_tokens)
                sentence_to_chunk_map.append([current_chunk_idx])
                current_chunk_idx += 1
            else:
                sent_chunks_idx = []
                for i in range(0, len(tokens), self.max_length - self.stride - 2):
                    window_tokens = tokens[i : i + (self.max_length - 2)]
                    chunk_tokens = [tokenizer.cls_token_id] + window_tokens + [tokenizer.sep_token_id]
                    chunked_inputs.append(chunk_tokens)
                    sent_chunks_idx.append(current_chunk_idx)
                    current_chunk_idx += 1
                sentence_to_chunk_map.append(sent_chunks_idx)

        chunk_embeddings = []
        model.eval()
        with torch.no_grad():
            for i in range(0, len(chunked_inputs), self.batch_size):
                batch_tokens = chunked_inputs[i : i + self.batch_size]

                max_len = max(len(t) for t in batch_tokens)
                padded_batch = [t + [tokenizer.pad_token_id] * (max_len - len(t)) for t in batch_tokens]
                attention_mask = [[1 if token != tokenizer.pad_token_id else 0 for token in t] for t in padded_batch]

                input_ids = torch.tensor(padded_batch).to(self.device)
                attention_mask = torch.tensor(attention_mask).to(self.device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                embeddings = self._mean_pooling(outputs, attention_mask)

                embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
                chunk_embeddings.extend(embeddings.cpu().numpy())

        for chunk_indices in sentence_to_chunk_map:
            sent_embeds = [chunk_embeddings[idx] for idx in chunk_indices]
            final_sent_embed = np.mean(sent_embeds, axis=0)
            final_sent_embed = final_sent_embed / np.linalg.norm(final_sent_embed)
            all_sentence_embeddings.append(final_sent_embed)

        return np.array(all_sentence_embeddings)

    def execute(self, df, text_col, label_col=None, **kwargs):
        tqdm.write("\n" + "="*60)
        tqdm.write("GIAI ĐOẠN 7: Trích xuất đặc trưng Embedded PhoBERT")
        tqdm.write("="*60)

        tqdm.write("[1/3] Trích xuất POS, NER Overlap và tách câu (CPU)...")
        tqdm.pandas(desc="[CPU Extraction]")
        cpu_results = df[text_col].progress_apply(self._compute_cpu_features)

        df['pos_distribution_variance'] = cpu_results.apply(lambda x: x[0])
        df['entity_grid_score'] = cpu_results.apply(lambda x: x[1])

        all_sentences = []
        text_sent_indices = []
        current_idx = 0

        for sents in cpu_results.apply(lambda x: x[2]):
            n_sents = len(sents)
            text_sent_indices.append((current_idx, current_idx + n_sents))
            all_sentences.extend(sents)
            current_idx += n_sents

        tqdm.write(f"\n[2/3] Xử lý GPU ({len(all_sentences)} câu) với batch_size={self.batch_size}...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            model = AutoModel.from_pretrained(self.model_name).to(self.device)

            all_embeddings = self._get_sentence_embeddings(all_sentences, tokenizer, model)

            del model
            del tokenizer
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e:
            tqdm.write(f"Lỗi khởi tạo/chạy GPU: {e}")
            for col in ['sentence_embedding_coherence', 'topic_drift_score', 'semantic_drift_score', 'embedding_distribution_entropy']:
                df[col] = 0.0
            return df

        tqdm.write("\n[3/3] Tính toán ngữ nghĩa...")
        coherences, topic_drifts, sem_drifts, entropies = [], [], [], []

        from sklearn.metrics.pairwise import cosine_similarity

        for start, end in tqdm(text_sent_indices, desc="[Semantic Calc]"):
            n = end - start
            if n < 2:
                coherences.append(0.0)
                topic_drifts.append(0.0)
                sem_drifts.append(0.0)
                entropies.append(0.0)
                continue

            doc_embeds = all_embeddings[start:end]

            sims = [cosine_similarity([doc_embeds[i]], [doc_embeds[i+1]])[0][0] for i in range(n-1)]
            coherences.append(float(np.mean(sims)))
            topic_drifts.append(float(np.var(sims)))

            sem_drift = 1.0 - cosine_similarity([doc_embeds[0]], [doc_embeds[-1]])[0][0]
            sem_drifts.append(float(sem_drift))

            try:
                centered_embeds = doc_embeds - np.mean(doc_embeds, axis=0)
                _, S, _ = np.linalg.svd(centered_embeds, full_matrices=False)

                explained_variance = (S ** 2) / (n - 1)
                total_var = np.sum(explained_variance)

                if total_var > 0:
                    probs = explained_variance / total_var
                    probs = probs[probs > 0]
                    entropy_val = -np.sum(probs * np.log2(probs))
                else:
                    entropy_val = 0.0
                entropies.append(float(entropy_val))
            except:
                entropies.append(0.0)

        df['sentence_embedding_coherence'] = coherences
        df['topic_drift_score'] = topic_drifts
        df['semantic_drift_score'] = sem_drifts
        df['embedding_distribution_entropy'] = entropies

        tqdm.write("\n[Hoàn tất] Pipeline hợp nhất chạy thành công!")
        return df

### 5. Đặc trưng Tương tác (InteractionFeatureStep)

**Mục đích:** Kết hợp các chỉ số đơn lẻ để tìm ra các mẫu (patterns) phức tạp hơn.

**Cơ chế:** Nhân/chia các đặc trưng (ví dụ: độ biến thiên GPT x độ trôi dạt ngữ nghĩa) để khuếch đại tín hiệu phân loại giữa Người và AI.

In [ ]:
class InteractionFeatureStep:
    def execute(self, df: pd.DataFrame, text_col: str, label_col: str) -> pd.DataFrame:
        print("\n")
        print("="*60)
        print("="*60)

        out_df = df.copy()

        out_df['gpt_var_x_semantic_drift'] = out_df['gpt_var'] * out_df['semantic_drift_score']
        out_df['gpt_mean_x_embedding_coherence'] = out_df['gpt_mean'] * out_df['sentence_embedding_coherence']
        out_df['gpt_tail_x_semantic_drift'] = out_df['gpt_tail'] * out_df['semantic_drift_score']

        out_df['sent_len_cv_x_gpt_var'] = out_df['sent_len_cv'] * out_df['gpt_var']
        out_df['punctuation_density_x_gpt_mean'] = out_df['punctuation_density'] * out_df['gpt_mean']
        out_df['comma_density_x_gpt_var'] = out_df['comma_density'] * out_df['gpt_var']

        out_df['logic_density_x_semantic_drift'] = out_df['logic_density'] * out_df['semantic_drift_score']
        out_df['logic_deviation_x_embedding_coherence'] = out_df['logic_deviation'] * out_df['sentence_embedding_coherence']

        out_df['transition_density'] = (out_df['causal_density'] +
                                        out_df['contrast_density'] +
                                        out_df['additive_density'] +
                                        out_df['temporal_density'])
        out_df['transition_density_x_gpt_var'] = out_df['transition_density'] * out_df['gpt_var']

        out_df['embedding_coherence_x_entropy'] = out_df['sentence_embedding_coherence'] * out_df['embedding_distribution_entropy']
        out_df['semantic_drift_x_entropy'] = out_df['semantic_drift_score'] * out_df['embedding_distribution_entropy']

        out_df['semantic_drift_ratio'] = out_df['semantic_drift_score'] / (out_df['gpt_var'] + 1e-5)
        out_df['sent_len_cv_ratio'] = out_df['sent_len_cv'] / (out_df['gpt_mean'] + 1e-5)

        out_df['combo_gpt_sent_punct'] = out_df['gpt_var'] * out_df['sent_len_cv'] * out_df['punctuation_density']

        print("Hoàn thành tính toán các đặc trưng tương tác.")
        return out_df

### TỔNG HỢP DANH SÁCH ĐẶC TRƯNG

Dưới đây là danh sách toàn bộ các đặc trưng đã được trích xuất trong pipeline, chia làm hai nhóm chính:

#### 1. Nhóm đặc trưng Truyền thống
*Các chỉ số này được tính toán dựa trên thống kê, từ điển hoặc quy tắc ngôn ngữ thuần túy, không thông qua mô hình Deep Learning.*

| Nhóm | Đặc trưng | Ý nghĩa |
| :--- | :--- | :--- |
| **Logic** | `logic_density`, `logic_mismatch_score`, `logic_deviation` | Mật độ từ nối (Biến từ CONST_SEED) và độ lệch so với trung bình domain. |
| **Nhịp điệu** | `sent_len_cv`, `sentence_transition_var` | Độ biến thiên chiều dài câu (Hệ số biến thiên) và sự thay đổi nhịp điệu giữa các câu. |
| **Dấu câu** | `comma_density`, `punctuation_density`, `comma_std_per_sentence` | Mật độ dấu phẩy/dấu câu và tính ổn định của việc dùng dấu câu. |
| **Từ loại** | `pronoun_ratio`, `pos_distribution_variance` | Tỷ lệ sử dụng đại từ nhân xưng và độ biến thiên của các loại từ (danh từ, động từ...). |

#### 2. Nhóm đặc trưng Lai Model
*Các chỉ số này được trích xuất từ đầu ra của các mô hình ngôn ngữ (GPT-Neo, PhoBERT) hoặc là kết quả của sự tương tác giữa chúng.*

| Nhóm | Đặc trưng | Ý nghĩa |
| :--- | :--- | :--- |
| **Xác suất (GPT)** | `gpt_mean`, `gpt_var`, `gpt_tail`, `gpt_d2_entropy` | Độ bất ngờ (Surprisal) và entropy của tốc độ thay đổi xác suất từ GPT-Neo. |
| **Ngữ nghĩa (BERT)** | `sentence_embedding_coherence`, `topic_drift_score`, `entity_grid_score` | Độ mạch lạc ngữ nghĩa, độ trôi dạt chủ đề và mạch thực thể từ PhoBERT. |

#### 3. Nhóm đặc trưng Tương tác (Interaction Features)
*Các đặc trưng này được tạo ra bằng cách kết hợp (nhân/chia) các chỉ số từ các bước trước đó nhằm phát hiện các mẫu hành vi phức tạp của AI.*

| Đặc trưng | Thành phần kết hợp | Lý do tạo và ý nghĩa trong phân loại |
| :--- | :--- | :--- |
| `gpt_var_x_semantic_drift` | GPT Variance & Semantic Drift | **Khuếch đại lỗi 'vô hồn':** AI thường có biến thiên xác suất thấp (viết đều đều) kèm theo sự trôi dạt ngữ nghĩa (mất dấu mục tiêu ban đầu). Việc nhân hai chỉ số này giúp làm nổi bật các đoạn văn vừa máy móc vừa thiếu nhất quán. |
| `sent_len_cv_x_gpt_var` | Sentence CV & GPT Variance | **Nhận diện 'Nhịp điệu máy':** Kết hợp sự ổn định về độ dài câu và sự ổn định về xác suất từ. Đây là 'vân tay' điển hình của AI: cấu trúc câu đều đặn và lựa chọn từ ngữ an toàn. |
| `logic_density_x_semantic_drift` | Logic Density & Semantic Drift | **Bắt lỗi 'Logic giả':** AI thường dùng nhiều từ nối (density cao) để tạo cảm giác logic nhưng nội dung thực tế lại bị trôi xa khỏi chủ đề ban đầu (drift cao). |
| `semantic_drift_ratio` | Semantic Drift / GPT Var | **Tỷ lệ bất thường:** So sánh tốc độ trôi dạt nội dung với độ biến thiên từ vựng. Người viết có thể trôi ý nhưng thường đi kèm với sự thay đổi cảm xúc/từ vựng mạnh mẽ, trong khi AI trôi ý một cách 'phẳng lặng'. |
| `combo_gpt_sent_punct` | GPT Var, Sent CV, Punct Density | **Chỉ số 'Tam giác vàng':** Tổng hợp 3 tín hiệu mạnh nhất: độ biến thiên từ ngữ, nhịp điệu câu và mật độ dấu câu. Đây là bộ lọc cuối cùng để tách biệt hoàn toàn văn bản có hồn của người viết. |

### Pipeline dành riêng cho deep learning

In [ ]:
class TextSegmentationStep(PipelineStep):
    def execute(self, df, text_col, label_col=None):
        print("\n" + "="*60)
        print("GIAI ĐOẠN PHÂN TÁCH TỪ (WORD SEGMENTATION)")
        print("="*60)
        print("[Segmentation] Đang tách từ bằng Underthesea (Đa luồng)...")

        df['segmented_text'] = df[text_col].parallel_apply(lambda x: word_tokenize(str(x), format="text"))

        print("Hoàn tất phân tách từ.")
        return df


In [ ]:
class DeepLearningExportStep(PipelineStep):
    def __init__(self, output_path):
        self.output_path = output_path

    def execute(self, df, text_col, label_col):
        print("\n" + "="*60)
        print("GIAI ĐOẠN DỌN DẸP VÀ XUẤT DỮ LIỆU DEEP LEARNING")
        print("="*60)

        required_cols = [text_col, 'segmented_text', label_col]
        cols_to_keep = [col for col in required_cols if col in df.columns]
        df_dl = df[cols_to_keep].copy()

        print("THÔNG TIN BỘ DỮ LIỆU DEEP LEARNING:")
        df_dl.info()
        print("\nXem thử 1 dòng dữ liệu Segmented:")
        print(df_dl['segmented_text'].iloc[0][:200] + "...")
        print(f"\nHOÀN TẤT PIPELINE DEEP LEARNING!")

        return df_dl

# Tổng kết pipeline

In [ ]:
class SummaryAndExportStep(PipelineStep):
    def __init__(self, output_path):
        self.output_path = output_path

    def execute(self, df, text_col, label_col):
        print("\n" + "="*60)
        print(" GIAI ĐOẠN DỌN DẸP, TỔNG KẾT VÀ XUẤT DỮ LIỆU")
        print("="*60)

        if 'seg_length' in df.columns:
            df = df.drop(columns=['seg_length'])

        print("THÔNG TIN BỘ DỮ LIỆU CUỐI CÙNG:")
        df.info()

        os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
        df.to_csv(self.output_path, index=False, encoding='utf-8-sig')
        print(f"\n HOÀN TẤT PIPELINE TỐI ƯU! File đã được lưu tại: {self.output_path}")

        return df

In [ ]:
class HuggingFaceUploadStep(PipelineStep):
    def __init__(self, repo_id: str, token: str = None):
        self.repo_id = repo_id
        self.token = token or os.environ.get("HF_TOKEN")

    def execute(self, df, text_col, label_col=None):
        print("\n" + "="*60)
        print(" GIAI ĐOẠN TÍCH HỢP: ĐẨY DỮ LIỆU LÊN HUGGING FACE HUB")
        print("="*60)

        if not self.token:
            print("Cảnh báo: Không tìm thấy Hugging Face Token. Vui lòng login bằng `huggingface-cli login` hoặc nạp vào file .env.")
            print("-> Đã bỏ qua bước upload.\n")
            return df

        try:
            print("[1/2] Đang chuyển đổi DataFrame sang Hugging Face Dataset...")
            df_cleaned = df.reset_index(drop=True)
            hf_dataset = Dataset.from_pandas(df_cleaned)

            print(f"[2/2] Đang đẩy dataset lên repository: {self.repo_id}...")
            hf_dataset.push_to_hub(
                repo_id=self.repo_id,
                token=self.token,
                private=False
            )

            print(f"\nHOÀN TẤT! Dataset đã được đưa lên mây tại:")
            print(f"https://huggingface.co/datasets/{self.repo_id}")

        except Exception as e:
            print(f"Đã xảy ra lỗi khi upload lên Hugging Face: {e}")
            print("Vui lòng kiểm tra lại Token (phải có quyền Write) hoặc kết nối mạng.")

        print("="*60)
        return df

In [ ]:
class EnhancedEDAPipeline:
    def __init__(self, text_col="text", label_col="label", checkpoint_base_dir="Data/checkpoints", run_id=None):
        self.text_col = text_col
        self.label_col = label_col
        self.steps = []
        self.checkpoint_base_dir = checkpoint_base_dir

        if run_id:
            self.run_id = run_id
            self.is_resume = True
        else:
            self.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
            self.is_resume = False

        self.current_ckpt_dir = os.path.join(self.checkpoint_base_dir, f"run_{self.run_id}")
        os.makedirs(self.current_ckpt_dir, exist_ok=True)

    def add_step(self, step):
        self.steps.append(step)

    def run(self, df=None):
        start_idx = 0
        current_df = df

        if self.is_resume:
            ckpt_files = glob.glob(os.path.join(self.current_ckpt_dir, "step_*.csv"))

            if ckpt_files:
                ckpt_files.sort(key=lambda x: int(os.path.basename(x).split('_')[1]))
                latest_ckpt_path = ckpt_files[-1]
                latest_step_num = int(os.path.basename(latest_ckpt_path).split('_')[1])

                print(f"[RESUME] Tìm thấy checkpoint gần nhất: {os.path.basename(latest_ckpt_path)}")
                print(f"[RESUME] Đang nạp dữ liệu vào bộ nhớ...")
                current_df = pd.read_csv(latest_ckpt_path)
                start_idx = latest_step_num
                print(f"[RESUME] Sẽ tự động bỏ qua {latest_step_num} bước đầu tiên. Bắt đầu từ bước {start_idx + 1}.\n")
            else:
                print(f"[CẢNH BÁO] Không tìm thấy file checkpoint nào trong {self.current_ckpt_dir}. Sẽ chạy lại từ đầu.")
                if df is None:
                    raise ValueError("Không có dữ liệu đầu vào. Vui lòng cung cấp 'df'.")

        if current_df is None and df is None:
            raise ValueError("Cần truyền dataframe đầu vào (df) cho lần chạy đầu tiên!")
        elif current_df is None:
             current_df = df.copy()

        for idx in range(start_idx, len(self.steps)):
            step = self.steps[idx]
            step_name = step.__class__.__name__
            step_num = idx + 1

            try:
                print(f"[Chạy Bước {step_num}/{len(self.steps)}] - {step_name}...")
                current_df = step.execute(current_df, self.text_col, self.label_col)

                ckpt_filename = f"step_{step_num}_{step_name}.csv"
                ckpt_path = os.path.join(self.current_ckpt_dir, ckpt_filename)

                current_df.to_csv(ckpt_path, index=False, encoding='utf-8-sig')
                print(f"[Đã lưu Checkpoint] -> {ckpt_filename}\n")

            except Exception as e:
                print(f"\nPIPELINE CRASH TẠI BƯỚC {step_num} ({step_name})")
                print(f"Chi tiết lỗi: {e}")
                print(f"Dữ liệu đã được bảo toàn ở bước {idx}. Vui lòng truyền tham số `run_id='{self.run_id}'` vào EnhancedEDAPipeline để chạy tiếp mà không cần sửa code.\n")
                break

        return current_df

### Chạy code

In [ ]:
input_file_path = "Data/vietnamese_news_ai_dataset_eda.csv"

base_filename = os.path.splitext(os.path.basename(input_file_path))[0]
clean_basename = base_filename.replace("_filtered", "")

eda_output_path = f"Data/{clean_basename}_integration.csv"
dl_output_path = f"Data/{clean_basename}_deep_learning.csv"

try:
    raw_df = pd.read_csv(input_file_path)
    raw_df = raw_df.drop(columns=['origin'], errors='ignore')
    raw_df = raw_df.rename(columns={'Text': 'text', 'Label': 'label'})
    pipeline = EnhancedEDAPipeline(
        text_col="text",
        label_col="label",
    )
    pipeline.add_step(BasicExplorationStep())
    pipeline.add_step(SafeNormalizationAndFilteringStep(min_words=150, max_words=600))
    pipeline.add_step(StratifiedSamplingStep(seed=2026))
    pipeline.add_step(AdvancedLogicFeatureExtractor(seeds=CONST_SEED))
    pipeline.add_step(CausalSurprisalStep())
    pipeline.add_step(SyntacticAndStructuralStep())
    pipeline.add_step(PhoBERTFeaturePipeline())
    pipeline.add_step(InteractionFeatureStep())
    pipeline.add_step(SummaryAndExportStep(output_path=eda_output_path))
    pipeline.add_step(HuggingFaceUploadStep(repo_id=HF_REPO_ID))
    final_df = pipeline.run(raw_df)

except FileNotFoundError:
    print("Vui lòng nạp đúng đường dẫn tới file CSV.")

🚀 [Chạy Bước 1/2] - InteractionFeatureStep...
Đang tính toán 15 đặc trưng tương tác (Interaction Features)...
Hoàn thành tính toán các đặc trưng tương tác.
✅ [Đã lưu Checkpoint] -> step_1_InteractionFeatureStep.csv

🚀 [Chạy Bước 2/2] - SummaryAndExportStep...

 GIAI ĐOẠN DỌN DẸP, TỔNG KẾT VÀ XUẤT DỮ LIỆU
THÔNG TIN BỘ DỮ LIỆU CUỐI CÙNG:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4012 entries, 0 to 4011
Data columns (total 45 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   text                                   4012 non-null   object 
 1   label                                  4012 non-null   int64  
 2   causal_density                         4012 non-null   float64
 3   contrast_density                       4012 non-null   float64
 4   additive_density                       4012 non-null   float64
 5   temporal_density                       4012 non-null   float64
 6   logic_